<a href="https://colab.research.google.com/github/Dubnitskyi/Machine_learning_labs/blob/main/lab3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install langchain-community langchain-core chromadb sentence-transformers

In [ ]:
import pandas as pd
import os
import time
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

In [ ]:
try:
  df = pd.read_csv('netflix_titles.csv')
except FileNotFoundError:
  print("Помилка: Файл netflix_titles.csv не знайдено.")

df = df.dropna(subset=['description']).drop_duplicates(subset=['title','description']).reset_index(drop=True)

print(df.describe(include='all'))

In [ ]:
df['decade'] = (df['release_year'] // 10) * 10

print("Added 'decade' column to df. First 5 rows with new column:")
print(df[['title', 'release_year', 'decade']].head())

In [ ]:
docs = []
for _, row in df.iterrows():
  metadata = {
    "title": str(row['title']),
    "year": int(row['release_year']),
    "type": str(row.get('type', 'Unknown')),
    "genre": [g.strip() for g in str(row.get('listed_in', 'Unknown')).split(',')],
    "rating": str(row.get('rating', 'Unknown')),
    "decade": int(row['decade'])
  }
  docs.append(Document(page_content=row['description'], metadata=metadata))

print(f"Дані очищено. Підготовлено {len(docs)} унікальних документів.")

In [ ]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=model_name)
# PersistentClient allows storing the database on disk
persist_directory = "./netflix_db_langchain"
# Create or load the vector store
vectorstore = Chroma.from_documents(
  documents=docs,
  embedding=embeddings,
  persist_directory=persist_directory,
  collection_metadata={"hnsw:space": "cosine"}
)
print("Vectorstore has been rebuilt with updated document metadata.")

# Task 1

In [ ]:
query = "Dramatic story about high school friendship and secrets"

print("\n--- Пошук за типом (Movie) ---")
results_type = vectorstore.similarity_search_with_score(
  query,
  k=3,
  filter={"$and":[{"type": {"$eq": "Movie"}},{"year": {"$gt": 2018}}]}
)
print(f"{'#':<3} | {'Назва':<25} | {'Схожість':<10} | {'Тип':<10} | {'Рік'}")
for i, (doc, score) in enumerate(results_type, 1):
  similarity = round(1 - score, 4)
  title = doc.metadata.get('title', 'N/A')
  item_type = doc.metadata.get('type', 'N/A')
  year = doc.metadata.get('year', 'N/A')
  print(f"{i:<3} | {title[:25]:<25} | {similarity:<10} | {item_type:<10} | {year}")
  print(f"  Опис: {doc.page_content[:100]}...")

In [ ]:
print("\n--- Комбінований пошук (рік > 2018, рейтинг TV-MA) ---")
results_combined = vectorstore.similarity_search_with_score(
  query,
  k=3,
  filter={"$and": [
    {"year": {"$gt": 2018}},
    {"rating": {"$eq": "TV-MA"}}
  ]}
)
print(f"{'#':<3} | {'Назва':<25} | {'Схожість':<10} | {'Рік':<10} | {'Рейтинг'}")
for i, (doc, score) in enumerate(results_combined, 1):
  similarity = round(1 - score, 4)
  title = doc.metadata.get('title', 'N/A')
  year = doc.metadata.get('year', 'N/A')
  rating = doc.metadata.get('rating', 'N/A')
  print(f"{i:<3} | {title[:25]:<25} | {similarity:<10} | {year:<10} | {rating}")
  print(f"  Опис: {doc.page_content[:100]}...")

## genre

In [ ]:
print("\n--- Пошук за жанром ---")
results_year_genre = vectorstore.similarity_search_with_score(
  query,
  k=3,
    filter={
        "genre": {"$nin": ['Comedies', 'Dramas']}
    }
)

print(f"{'#':<3} | {'Назва':<25} | {'Схожість':<10} | {'Рік':<10} | {'Жанр'}")
for i, (doc, score) in enumerate(results_year_genre, 1):
  similarity = round(1 - score, 4)
  title = doc.metadata.get('title', 'N/A')
  year = doc.metadata.get('year', 'N/A')
  genre = doc.metadata.get('genre', 'N/A')
  print(f"{i:<3} | {title[:25]:<25} | {similarity:<10} | {year:<10} | {str(genre)[:30]}")
  print(f"Опис: {doc.page_content[:100]}...\n")

In [ ]:
print("\n--- Пошук за рейтингом (TV-MA) та роком (>2015) ---")
results_rating_year = vectorstore.similarity_search_with_score(
  query,
  k=3,
  filter={
    "$and": [
      {"rating": {"$eq": "TV-MA"}},
      {"year": {"$gt": 2015}}
    ]
  }
)

print(f"{'#':<3} | {'Назва':<25} | {'Схожість':<10} | {'Рік':<10} | {'Рейтинг'}")
for i, (doc, score) in enumerate(results_rating_year, 1):
  similarity = round(1 - score, 4)
  title = doc.metadata.get('title', 'N/A')
  year = doc.metadata.get('year', 'N/A')
  rating = doc.metadata.get('rating', 'N/A')
  print(f"{i:<3} | {title[:25]:<25} | {similarity:<10} | {year:<10} | {rating}")
  print(f"  Опис: {doc.page_content[:100]}...")

In [ ]:
print("\n--- Reranking results by score (descending) ---")
# Perform a standard vector search to get a broader set of results
rerank_candidates = vectorstore.similarity_search_with_score(query, k=10)
# Sort these documents based on the 'score' metadata field in descending order
reranked_results = sorted(rerank_candidates, key=lambda x: x[1], reverse=True)

print(f"{'#':<3} | {'Назва':<25} | {'Схожість':<10} | {'Рік':<10} | {'Рейтинг'}")
for i, (doc, score) in enumerate(reranked_results, 1):
  similarity = round(1 - score, 4)
  title = doc.metadata.get('title', 'N/A')
  year = doc.metadata.get('year', 'N/A')
  rating = doc.metadata.get('rating', 'N/A')
  print(f"{i:<3} | {title[:25]:<25} | {similarity:<10} | {year:<10} | {rating}")
  print(f"  Опис: {doc.page_content[:100]}...")

In [ ]:
print("\n--- Аналіз ефективності запитів (час виконання) ---")

query = "Dramatic story about high school friendship and secrets"

# Вимірювання часу для невідфільтрованого пошуку
start_time = time.time()
unfiltered_results_time = vectorstore.similarity_search_with_score(query, k=10)
end_time = time.time()
unfiltered_duration = end_time - start_time
print(f"Час виконання невідфільтрованого пошуку: {unfiltered_duration:.4f} секунд")

# Вимірювання часу для відфільтрованого пошуку (наприклад, рік > 2018 та жанр 'Dramas')
start_time = time.time()
filtered_results_time = vectorstore.similarity_search_with_score(
  query,
  k=10,
  filter={"year": {"$gt": 2018}
  }
)
end_time = time.time()
filtered_duration = end_time - start_time
print(f"Час виконання відфільтрованого пошуку (рік > 2018): {filtered_duration:.4f} секунд")

print("\nПорівняння часу виконання:")
print(f"  Невідфільтрований пошук: {unfiltered_duration:.4f} с")
print(f"  Відфільтрований пошук: {filtered_duration:.4f} с")


# Task 2

## Обґрунтування вибору полів тексту для векторизації.

### Аналіз та Обґрунтування

Для створення ефективних векторних представлень для семантичного аналізу з DataFrame `df`, слід ретельно розглянути поля `title` та `description`.

*   **Поле 'title'**: Містить назву фільму або серіалу. Назви зазвичай короткі і, хоча вони можуть натякати на тематику, вони рідко надають достатньо детальної інформації для глибокого семантичного розуміння. Наприклад, назва 'Blood & Water' не дає повного уявлення про сюжет чи жанр.

*   **Поле 'description'**: Містить розширений опис сюжету, ключових подій, персонажів та загальної атмосфери фільму чи серіалу. Це поле є значно багатшим на контекстну інформацію.

## Аналіз та уточнення структури метаданих для жорсткої фільтрації


  *   `title`: Назва фільму/серіалу.
  *   `year`: Рік випуску (ціле число).
  *   `type`: Тип контенту ('Movie' або 'TV Show').
  *   `genre`: Список жанрів (розділені рядки з `listed_in`).
  *   `rating`: Віковий рейтинг.

## Розробка стратегії семантичних запитів.

### Semantic Queries for Testing:

1.  **Загальний запит**: "High school dramas"
    *   _Опис_: Широкий запит для отримання фільмів і серіалів про середню школу.
2.  **Дещо специфічний запит**: "Teenage friendship problems"
    *   _Опис_: Фокусується на проблемах у відносинах підлітків, але все ще досить загальний.
3.  **Специфічний запит (ключові слова)**: "Secrets in high school friendships"
    *   _Опис_: Використовує ключові терміни з основної теми.
4.  **Специфічний запит (з акцентом на емоції/драму)**: "Dramatic stories about betrayal among friends in high school"
    *   _Опис_: Включає емоційний аспект та специфіку конфлікту.
5.  **Дуже специфічний запит**: "Teenage friendships torn apart by dark secrets"
    *   _Опис_: Максимально близький до початкової теми з усіма ключовими елементами.
6.  **Запит з кількома концепціями**: "Coming-of-age stories with secrets and friendship drama"
    *   _Опис_: Комбінує кілька концепцій, які можуть бути релевантними.
7.  **Запит, орієнтований на наслідки**: "High school students dealing with consequences of hidden truths"
    *   _Опис_: Фокусується на результатах або розвитку сюжету, пов'язаного із секретами.

In [ ]:
semantic_queries = [
    "High school dramas",
    "Teenage friendship problems",
    "Secrets in high school friendships",
    "Dramatic stories about betrayal among friends in high school",
    "Teenage friendships torn apart by dark secrets",
    "Coming-of-age stories with secrets and friendship drama",
    "High school students dealing with consequences of hidden truths"
]

N_results = 5

print("\n--- Conducting Control Semantic Queries ---")

for i, query in enumerate(semantic_queries):
    print(f"\nQuery {i+1}: {query}")
    print("---------------------------------------------------")

    # Perform vector search
    results = vectorstore.similarity_search_with_score(query, k=N_results)

    print(f"{'#':<3} | {'Назва':<30} | {'Схожість':<10} | {'Рік':<5} | {'Тип':<10} | {'Рейтинг'}")
    for j, (doc, score) in enumerate(results, 1):
        similarity = round(1 - score, 4)
        title = doc.metadata.get('title', 'N/A')
        year = doc.metadata.get('year', 'N/A')
        item_type = doc.metadata.get('type', 'N/A')
        rating = doc.metadata.get('rating', 'N/A')
        print(f"{j:<3} | {title[:30]:<30} | {similarity:<10} | {year:<5} | {item_type:<10} | {rating}")
        print(f"Опис: {doc.page_content[:100]}...")


## Аналіз результатів семантичних запитів та оцінка ефективності моделі

#### Аналіз релевантності та косинусної схожості

**Query 1: "High school dramas"**
*   **Results**: `Westside Story` (TV Show, 2003), `Hard Lessons` (Movie, 1986), `Age of Rebellion` (TV Show, 2018), `Moesha` (TV Show, 2000), `The Underclass` (TV Show, 2020).
*   **Relevance**: Всі отримані результати є релевантними, оскільки їх описи чітко вказують на тематику середньої школи та драматичний сюжет. Схожість коливається від 0.59 до 0.54. Найвищий показник у `Westside Story` (0.5937), що є очікуваним.
*   **Semantic Noise**: Мінімальний. Загалом, результати відповідають широкому запиту.

**Query 2: "Teenage friendship problems"**
*   **Results**: `Mismatched` (TV Show, 2020), `On My Block` (TV Show, 2020), `The Dreamer` (Movie, 2009), `The Edge of Seventeen` (Movie, 2016), `Banana Split` (Movie, 2020).
*   **Relevance**: Результати дуже релевантні. Описи фільмів `Mismatched` ("strike up a tentative friendship"), `On My Block` ("find their li..."), `The Edge of Seventeen` ("finds high school life even less bearable after she catches her childhood best f...") прямо вказують на проблеми у відносинах підлітків. Схожість висока, особливо у `Mismatched` (0.672).
*   **Semantic Noise**: Незначний. `The Dreamer` і `Banana Split` також підходять, хоча менш прямолінійно, ніж перші два.

**Query 3: "Secrets in high school friendships"**
*   **Results**: `Tango` (TV Show, 2018), `Unrequited Love` (TV Show, 2019), `On My Block` (TV Show, 2020), `Mismatched` (TV Show, 2020), `Face 2 Face` (Movie, 2017).
*   **Relevance**: Всі результати є дуже релевантними. `Tango` ("Shocking secrets begin to unravel..."), `Unrequited Love` ("one-sided secret attraction"), `Face 2 Face` ("learn about each other's insecurities and secrets") безпосередньо стосуються "секретів у дружбі". Схожість від 0.62 до 0.55.
*   **Semantic Noise**: Дуже низький. Модель чудово вловила ключові концепції запиту.

**Query 4: "Dramatic stories about betrayal among friends in high school"**
*   **Results**: `Pyaar Tune Kya Kiya` (TV Show, 2014), `Detention` (TV Show, 2020), `Trinkets` (TV Show, 2020), `Dead Kids` (Movie, 2019), `Get Even` (TV Show, 2020).
*   **Relevance**: Висока релевантність. `Detention` ("uncovers unsettling secrets at her remote high sc..."), `Trinkets` ("grieving teen finds an unexpected connection..."), `Get Even` ("In a secret act of skillful revenge, four private school classmates team up...") чітко відповідають концепції драми, зради та шкільного середовища. `Pyaar Tune Kya Kiya` також стосується підліткових драм.
*   **Semantic Noise**: Мінімальний. `Dead Kids` має трохи меншу пряму відповідність "зраді", але все ще вписується в "драматичні історії" зі шкільного життя.

**Query 5: "Teenage friendships torn apart by dark secrets"**
*   **Results**: `Mismatched` (TV Show, 2020), `Tango` (TV Show, 2018), `At First Light` (Movie, 2018), `The Dreamer` (Movie, 2009), `Tellur Aliens` (Movie, 2016).
*   **Relevance**: `Mismatched` і `Tango` знову демонструють високу релевантність, що підтверджує їх відповідність темі секретів і дружби. `At First Light` і `The Dreamer` також мають елементи підліткових проблем. `Tellur Aliens` є прикладом `semantic noise`.
*   **Semantic Noise**: Присутній у `Tellur Aliens` (0.5944), який, судячи з опису ("Three teenaged best friends on the planet Telluria begin an incredible journey of adventure..."), є науковою фантастикою і не стосується "темних секретів, що руйнують дружбу", хоча містить "teenaged best friends". Це показує, що іноді загальні терміни можуть збити модель, якщо вони є в описі, але не відображають основної теми.

**Query 6: "Coming-of-age stories with secrets and friendship drama"**
*   **Results**: `Pyaar Tune Kya Kiya` (TV Show, 2014), `What Are the Odds?` (Movie, 2020), `Cloud Atlas` (Movie, 2012), `Edge of Seventeen` (Movie, 1998), `The Tree of Blood` (Movie, 2018).
*   **Relevance**: `Pyaar Tune Kya Kiya` та `Edge of Seventeen` (1998) добре відповідають запиту, оскільки їх описи стосуються дорослішання та підліткових драм. `What Are the Odds?` також може бути релевантним. `Cloud Atlas` та `The Tree of Blood` мають нижчу релевантність.
*   **Semantic Noise**: `Cloud Atlas` (0.5822) є явним "шумом". Хоча це "drama" зі "complicated links that human...", вона не є "coming-of-age" і не про "secrets and friendship drama" у підлітковому контексті. Це вказує на те, що модель може іноді узагальнювати "drama" без достатнього контексту.

**Query 7: "High school students dealing with consequences of hidden truths"**
*   **Results**: `100 Things to do Before High S` (Movie, 2014), `ThirTEEN Terrors` (TV Show, 2014), `Girl's Revenge` (Movie, 2020), `The Witch: Part 1 - The Subver` (Movie, 2018), `Detention` (TV Show, 2020).
*   **Relevance**: `ThirTEEN Terrors` ("searches for the dark truth behind their school's mysterious..."), `Girl's Revenge` ("sets out to reveal..."), `Detention` ("uncovers unsettling secrets at her remote high school...") дуже релевантні, прямо відображаючи тему прихованих істин та їх наслідків. `The Witch` також має елементи таємниць та наслідків.
*   **Semantic Noise**: `100 Things to do Before High S` (0.5435) є "шумом", оскільки це комедійний серіал для молодших підлітків про "things to do before high school" і не має відношення до "consequences of hidden truths".

#### Висновок про ефективність щільних векторних представлень

Загалом, використання щільних векторних представлень (HuggingFaceEmbeddings) у поєднанні з ChromaDB є **дуже ефективним** для вирішення задачі семантичного пошуку та рекомендації контенту на основі описів фільмів та серіалів. Модель демонструє вражаючу здатність розуміти нюанси запитів, такі як "dramas", "friendship problems", "secrets", "betrayal" та "consequences of hidden truths", і знаходити відповідні описи, що містять ці концепції.

# Task 3

In [ ]:
target_concept = 'protagonist'
N_documents_per_decade = 15

unique_decades = sorted(df['decade'].unique())

decade_documents = {}

print(f"Performing semantic search for target concept: '{target_concept}' across {len(unique_decades)} decades...")

for decade in unique_decades:
    print(f"\n--- Searching for decade: {decade} ---")
    try:
        # Perform semantic similarity search with filter for the current decade
        # Explicitly cast decade to Python int to avoid type issues with ChromaDB filter
        results_for_decade = vectorstore.similarity_search_with_score(
            target_concept,
            k=N_documents_per_decade,
            filter={'decade': {'$eq': int(decade)}}
        )
        decade_documents[decade] = results_for_decade
        print(f"Collected {len(results_for_decade)} documents for decade {decade}.")
    except Exception as e:
        print(f"Error searching for decade {decade}: {e}")

print("\nSemantic search by decade complete.")

In [ ]:
import nltk
from collections import Counter

nltk.download('stopwords')

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

# Explicitly download punkt_tab as suggested by the error message if word_tokenize fails
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')

print("NLTK punkt and stopwords resources are available.")

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Initialize English stopwords
stop_words = set(stopwords.words('english'))

decade_frequent_terms = {}

print("\n--- Analyzing frequent terms per decade ---")

for decade, documents_with_scores in decade_documents.items():
    all_words_in_decade = []
    for doc, score in documents_with_scores:
        description = doc.page_content

        # Tokenize and convert to lowercase
        tokens = word_tokenize(description.lower())

        # Filter out stop words, non-alphabetic tokens, and short words
        filtered_tokens = [word for word in tokens if word.isalpha() and word not in stop_words and len(word) > 1]
        all_words_in_decade.extend(filtered_tokens)

    # Count word frequencies
    word_counts = Counter(all_words_in_decade)

    # Get the top 10 most frequent terms
    top_10_terms = word_counts.most_common(10)
    decade_frequent_terms[decade] = top_10_terms

    print(f"\nDecade: {decade}")
    print(f"  Top 10 frequent terms: {top_10_terms}")

print("\nFrequent terms analysis complete.")

## Summarize Semantic Evolution

Analyzing the most frequent terms associated with the 'protagonist' concept across different decades reveals interesting shifts and continuities in how central characters are portrayed in cinematic narratives.

**Early Decades (1920s - 1970s): Focus on External Conflict and Archetypes**

*   **1920s**: The single entry shows terms like 'collection', 'films', 'women', 'issues', 'norms', 'mark', suggesting that early protagonists, particularly women, were often involved in societal issues, leaving a significant impact. This hints at social commentary or pioneering roles.
*   **1940s**: Heavily dominated by 'war', 'documentary', 'film', 'world', 'combat', 'nazi', 'germany', 'italy'. This clearly reflects the World War II era, where protagonists were likely soldiers, war heroes, or civilians grappling with global conflict. The terms are very concrete and externally driven.
*   **1950s**: Terms like 'woman', 'falls', 'planet', 'finds', 'gigi' suggest a move towards more personal stories, potentially romantic narratives or science fiction exploring new worlds. 'Honest', 'man', 'dreams' also appear, hinting at simpler, perhaps idealistic, character traits.
*   **1960s**: 'Woman', 'young', 'man', 'new', 'showdown', 'becomes', 'danger', 'epic', 'leads', 'way'. This decade continues the focus on 'young' men and women, often encountering 'danger' and 'showdown', embarking on 'epic' journeys, and 'leading the way', indicating heroic or transformative arcs.
*   **1970s**: 'Family', 'village', 'dark', 'director', 'seeking', 'island', 'police', 'shark', 'vanishes'. The emergence of 'dark' themes, 'police', and elements like 'shark' (likely influenced by films like Jaws) points to protagonists facing more menacing, often isolated, threats within family or community settings. The term 'seeking' suggests characters on a quest or in search of answers.

**Middle Decades (1980s - 2000s): Rise of Action, Revenge, and Personal Struggle**

*   **1980s**: 'Man', 'young', 'revenge', 'skills', 'powerful', 'trying', 'cop', 'woman', 'brutal', 'rising'. This era clearly emphasizes 'man' as a central figure, often driven by 'revenge', equipped with specific 'skills', and confronting 'powerful' and 'brutal' adversaries. The 'cop' archetype also gains prominence. This reflects the rise of action heroes.
*   **1990s**: 'Life', 'crime', 'stop', 'gangster', 'take', 'boy', 'revenge', 'george', 'dennis'. While 'revenge' persists, 'crime' and 'gangster' themes become more pronounced, suggesting protagonists caught in criminal underworlds or working to 'stop' them. 'Life' as a frequent term might hint at broader life struggles within these contexts.
*   **2000s**: 'Man', 'must', 'thief', 'become', 'one', 'gangster', 'businessman', 'head', 'discovers', 'world'. The protagonist is often a 'man' who 'must' achieve something, sometimes a 'thief' or 'gangster', or even a 'businessman'. The term 'discovers' suggests internal or external revelations. The concept of 'world' implies a larger sphere of influence or conflict for the character.

**Recent Decades (2010s - 2020s): Complexity, Identity, and Personal Journey in a Global Context**

*   **2010s**: 'Life', 'discovers', 'man', 'fictional', 'living', 'best', 'wealth', 'young', 'gangster', 'trapped'. 'Life' and 'man' remain central. 'Discovers' continues from the 2000s. The appearance of 'fictional' and 'trapped' suggests more complex narratives, possibly meta-narratives or characters in confined situations. 'Wealth' and 'gangster' also persist, reflecting diverse societal roles.
*   **2020s**: 'Life', 'characters', 'young', 'man', 'world', 'new', 'gets', 'becomes', 'identity', 'seeking'. This decade sees 'life' and 'young' characters at the forefront. The explicit mention of 'characters' and 'identity' indicates a deeper focus on the internal aspects and development of individuals. 'New' and 'seeking' portray protagonists on journeys of self-discovery or adaptation to changed circumstances within a broader 'world'.

**Conclusion:**

The semantic evolution of the 'protagonist' concept reflects broader societal and narrative trends. While early portrayals were heavily influenced by historical events and external struggles, later decades saw a pivot towards more complex, internal explorations of identity, personal challenges, and moral ambiguities. Despite these shifts, the fundamental elements of human experience, personal growth, and confronting adversity remain continuous threads in the portrayal of protagonists over time. Dense vector representations effectively capture these nuanced changes and continuities, providing a valuable tool for analyzing cultural narratives.